# Varasana Detection Tutorial

**Purpose**: Demonstrate how to use the FCOS-based Candida morphology detector  
**Author**: Hallett Lab  
**Date**: 2026-01-28

This notebook walks through the complete detection workflow:
1. Environment setup and imports
2. Loading the pre-trained FCOS detector
3. Running inference on microscopy images
4. Extracting cell crops for VAE analysis
5. Statistical analysis of detections
6. Visualization of results

## Environment

```bash
conda activate candescence_new
```

**Note**: This module requires MMDETECTION. Install with:
```bash
pip install mmdet mmcv-full
```

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path for development (not needed if package is installed)
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Verify imports work
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Enable inline plotting
%matplotlib inline

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Candescence imports
from candescence.core.logging_config import get_logger, configure_logging
from candescence.core.morphology import (
    CLASS_NAMES,
    CLASS_TO_INDEX,
    CURRICULUM_STAGES,
    FILAMENTOUS_GRADES,
    NON_FILAMENTOUS_GRADES,
)

# Setup logging
configure_logging(level='INFO')
logger = get_logger("tutorial")

print("All imports successful!")
print(f"\nMorphology classes ({len(CLASS_NAMES)}):")
for i, name in enumerate(CLASS_NAMES):
    filamentous = "[F]" if name in FILAMENTOUS_GRADES else "   "
    print(f"  {i:2d}: {filamentous} {name}")

## 2. Morphology Class Overview

The detector recognizes 15 Candida morphology classes, organized into curriculum learning stages:

In [ ]:
# Display curriculum learning stages
print("Curriculum Learning Stages:")
print("=" * 60)
for stage in CURRICULUM_STAGES:
    print(f"\nStage: {stage.name.upper()}")
    print(f"  New classes: {stage.classes}")
    print(f"  Cumulative: {len(stage.cumulative_classes)} classes")

In [ ]:
# Visualize class hierarchy
fig, ax = plt.subplots(figsize=(12, 6))

# Create data for visualization
categories = {
    'Yeast': ['Yeast White', 'Yeast Opaque', 'Yeast Gray'],
    'Budding': ['Budding White', 'Budding Opaque', 'Budding Gray'],
    'Transitional': ['Shmoo'],
    'Filamentous': ['Pseudohyphae', 'Hyphae'],
    'Junctions': ['P-junction', 'H-junction', 'P-Start', 'H-Start'],
    'Other': ['Artifact', 'Unknown'],
}

colors = {
    'Yeast': '#90EE90',
    'Budding': '#98FB98', 
    'Transitional': '#FFD700',
    'Filamentous': '#87CEEB',
    'Junctions': '#DDA0DD',
    'Other': '#D3D3D3',
}

y_pos = 0
for cat, classes in categories.items():
    ax.barh([y_pos], [len(classes)], color=colors[cat], label=cat, height=0.6)
    ax.text(len(classes) + 0.1, y_pos, f"{', '.join(classes)}", va='center', fontsize=9)
    y_pos += 1

ax.set_yticks(range(len(categories)))
ax.set_yticklabels(list(categories.keys()))
ax.set_xlabel('Number of Classes')
ax.set_title('Candida Morphology Class Hierarchy')
ax.set_xlim(0, 8)
plt.tight_layout()
display(fig)
plt.close(fig)

## 3. Loading the Pre-trained Detector

The production FCOS detector is trained with curriculum learning on all 15 classes.

In [ ]:
# Check if production model exists
production_config = Path("<legacy-refined>/candescence_master/production/varasana/config.py")
production_model = Path("<legacy-refined>/candescence_master/production/varasana/model.pth")

print("Production Model Status:")
print(f"  Config: {production_config}")
print(f"    Exists: {production_config.exists()}")
print(f"  Model: {production_model}")
print(f"    Exists: {production_model.exists()}")
if production_model.exists():
    size_mb = production_model.stat().st_size / (1024 * 1024)
    print(f"    Size: {size_mb:.1f} MB")

In [ ]:
# Load the detector
# NOTE: This requires MMDETECTION to be installed
# If not installed, the following cells will demonstrate the API without running

try:
    from candescence.detection import CandidaDetector
    
    detector = CandidaDetector()
    print(f"Detector loaded successfully!")
    print(f"  Classes: {detector.num_classes}")
    print(f"  Device: {detector.device}")
    DETECTOR_AVAILABLE = True
    
except ImportError as e:
    print(f"MMDETECTION not available: {e}")
    print("\nTo install MMDETECTION:")
    print("  pip install mmdet mmcv-full")
    print("\nContinuing with demonstration mode...")
    DETECTOR_AVAILABLE = False
    detector = None

## 4. Running Detection on Images

The detector returns bounding boxes with class labels and confidence scores.

In [ ]:
# Find sample images to test with
# These would typically be DIC microscopy images of Candida plates

sample_image_dirs = [
    Path("data/sample/images"),
    Path("<legacy-refined>/candescence/train-data/final/val"),
    Path("/home/data/refined/candescence/train-data/final/val"),
]

sample_images = []
for img_dir in sample_image_dirs:
    if img_dir.exists():
        sample_images.extend(list(img_dir.glob("*.jpg"))[:5])
        sample_images.extend(list(img_dir.glob("*.png"))[:5])
        sample_images.extend(list(img_dir.glob("*.bmp"))[:5])

print(f"Found {len(sample_images)} sample images")
if sample_images:
    print("Sample paths:")
    for img in sample_images[:3]:
        print(f"  {img}")

In [ ]:
# Demonstrate detection API (works with or without actual detector)

if DETECTOR_AVAILABLE and sample_images:
    # Run actual detection
    test_image = sample_images[0]
    print(f"Running detection on: {test_image}")
    
    detections = detector.detect(test_image, threshold=0.5)
    
    print(f"\nFound {len(detections)} detections:")
    for det in detections[:10]:  # Show first 10
        print(f"  {det.label_name}: {det.score:.3f} at ({det.x1:.0f}, {det.y1:.0f}, {det.x2:.0f}, {det.y2:.0f})")
    if len(detections) > 10:
        print(f"  ... and {len(detections) - 10} more")
else:
    # Demonstrate API with mock data
    print("Demonstration mode - showing API usage:")
    print()
    print("from candescence.detection import CandidaDetector")
    print()
    print("# Load detector (uses production model by default)")
    print("detector = CandidaDetector()")
    print()
    print("# Run detection")
    print("detections = detector.detect('plate_image.jpg', threshold=0.5)")
    print()
    print("# Each detection has:")
    print("#   - bbox: (x1, y1, x2, y2)")
    print("#   - label: class index (0-14)")
    print("#   - label_name: 'Yeast White', 'Hyphae', etc.")
    print("#   - score: confidence (0-1)")

In [ ]:
# Visualize detections

if DETECTOR_AVAILABLE and sample_images:
    # Visualize with bounding boxes
    annotated = detector.visualize(
        test_image,
        threshold=0.5,
        show=False
    )
    
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(annotated[:, :, ::-1])  # BGR to RGB
    ax.set_title(f"Detections: {len(detections)} cells")
    ax.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close(fig)
else:
    print("Visualization requires detector and sample images")

## 5. Extracting Cell Crops for VAE Analysis

The `CellExtractor` crops detected cells and resizes them to a standard size for downstream VAE analysis.

In [ ]:
# Demonstrate cell extraction

if DETECTOR_AVAILABLE and sample_images:
    from candescence.detection import CellExtractor
    
    # Create extractor with 128x128 output (standard for VAE)
    extractor = CellExtractor(
        detector=detector,
        output_size=128,
        grayscale=False,
        padding=0.1  # Add 10% padding around bbox
    )
    
    # Extract cells
    crops = extractor.extract(test_image, threshold=0.5)
    
    print(f"Extracted {len(crops)} cell crops")
    if crops:
        print(f"Crop shape: {crops[0].shape}")
        print(f"\nClass distribution:")
        from collections import Counter
        class_counts = Counter(c.label_name for c in crops)
        for cls, count in class_counts.most_common():
            print(f"  {cls}: {count}")
else:
    print("Demonstration mode - showing API usage:")
    print()
    print("from candescence.detection import CellExtractor")
    print()
    print("# Create extractor")
    print("extractor = CellExtractor(")
    print("    detector=detector,")
    print("    output_size=128,  # Standard for VAE")
    print("    grayscale=False,")
    print("    padding=0.1")
    print(")")
    print()
    print("# Extract crops")
    print("crops = extractor.extract('image.jpg', threshold=0.5)")
    print()
    print("# Each crop has:")
    print("#   - image: numpy array (128, 128, 3)")
    print("#   - label, label_name, score")
    print("#   - source_image, bbox")

In [ ]:
# Visualize extracted crops

if DETECTOR_AVAILABLE and sample_images and crops:
    # Show first 20 crops
    n_show = min(20, len(crops))
    cols = 5
    rows = (n_show + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
    axes = np.atleast_2d(axes)
    
    for idx, ax in enumerate(axes.flat):
        if idx < n_show:
            crop = crops[idx]
            ax.imshow(crop.image[:, :, ::-1])  # BGR to RGB
            ax.set_title(f"{crop.label_name}\n{crop.score:.2f}", fontsize=9)
        ax.axis('off')
    
    fig.suptitle(f"Extracted Cell Crops ({n_show} of {len(crops)})", fontsize=14)
    plt.tight_layout()
    display(fig)
    plt.close(fig)
else:
    print("Crop visualization requires detector and sample images")

In [ ]:
# Convert crops to PyTorch tensor for VAE input

if DETECTOR_AVAILABLE and sample_images and crops:
    # Convert to tensor
    tensor = extractor.crops_to_tensor(crops, normalize=True)
    
    print(f"Tensor shape: {tensor.shape}")
    print(f"Tensor dtype: {tensor.dtype}")
    print(f"Value range: [{tensor.min():.3f}, {tensor.max():.3f}]")
    print()
    print("Ready for VAE input!")
else:
    print("Demonstration:")
    print()
    print("# Convert crops to tensor for VAE")
    print("tensor = extractor.crops_to_tensor(crops, normalize=True)")
    print("# Shape: (N, C, H, W) = (num_crops, 3, 128, 128)")

## 6. Statistical Analysis

The `MorphologyStats` class provides tools for analyzing detection results.

In [ ]:
from candescence.analysis import MorphologyStats
from candescence.detection.inference.detector import Detection

# Create stats collector
stats = MorphologyStats()

if DETECTOR_AVAILABLE and sample_images:
    # Add detections from our test image
    stats.add_detections(test_image, detections)
    
    # Process more images if available
    for img_path in sample_images[1:5]:
        dets = detector.detect(img_path, threshold=0.5)
        stats.add_detections(img_path, dets)
        print(f"Processed {img_path.name}: {len(dets)} detections")
else:
    # Create mock detections for demonstration
    mock_detections = [
        Detection(bbox=(10, 10, 50, 50), label=0, label_name="Yeast White", score=0.95),
        Detection(bbox=(60, 60, 100, 100), label=1, label_name="Budding White", score=0.87),
        Detection(bbox=(110, 110, 150, 150), label=10, label_name="Hyphae", score=0.92),
        Detection(bbox=(160, 160, 200, 200), label=9, label_name="Pseudohyphae", score=0.78),
        Detection(bbox=(210, 210, 250, 250), label=6, label_name="Shmoo", score=0.85),
    ]
    stats.add_detections(Path("mock_image_1.jpg"), mock_detections)
    stats.add_detections(Path("mock_image_2.jpg"), mock_detections[:3])
    print("Using mock data for demonstration")

In [ ]:
# Get summary statistics
summary = stats.summary()

print("Detection Summary")
print("=" * 50)
print(f"Total images processed: {summary['total_images']}")
print(f"Total detections: {summary['total_detections']}")
print(f"Detections per image: {summary['detections_per_image_mean']:.1f} ± {summary['detections_per_image_std']:.1f}")
print(f"\nConfidence statistics:")
print(f"  Mean: {summary['confidence_mean']:.3f}")
print(f"  Std:  {summary['confidence_std']:.3f}")
print(f"  Min:  {summary['confidence_min']:.3f}")
print(f"  Max:  {summary['confidence_max']:.3f}")
print(f"\nFilamentous ratio: {summary['filamentous_ratio']:.1%}")

In [ ]:
# Class distribution
class_df = stats.class_distribution()
print("Class Distribution:")
display(class_df)

In [ ]:
# Convert to DataFrame for detailed analysis
df = stats.to_dataframe()
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample rows:")
display(df.head())

## 7. Visualization

The `DetectionVisualizer` class provides plotting utilities.

In [ ]:
from candescence.analysis import DetectionVisualizer

viz = DetectionVisualizer(figsize=(10, 6))

# Get all detections for plotting
all_detections = [d for _, dets in stats._detections for d in dets]
print(f"Total detections for visualization: {len(all_detections)}")

In [ ]:
# Class distribution bar chart
fig = viz.plot_class_distribution(
    all_detections,
    title="Morphology Class Distribution",
    show_percentages=True
)
display(fig)
plt.close(fig)

In [ ]:
# Confidence histogram
fig = viz.plot_confidence_histogram(
    all_detections,
    bins=20,
    title="Detection Confidence Distribution"
)
display(fig)
plt.close(fig)

In [ ]:
# Size distribution
fig = viz.plot_size_distribution(
    all_detections,
    bins=20,
    title="Detection Size Distribution"
)
display(fig)
plt.close(fig)

In [ ]:
# Filamentous vs non-filamentous pie chart
fig = viz.plot_filamentous_pie(
    all_detections,
    title="Filamentous vs Non-Filamentous Morphologies"
)
display(fig)
plt.close(fig)

## 8. Integration with Tendril VAE

The extracted cell crops can be used directly with the Tendril VAE for latent space analysis.

In [ ]:
# Example: Complete pipeline from plate image to VAE encoding

print("Complete Detection → VAE Pipeline")
print("=" * 50)
print()
print("# Step 1: Detection")
print("from candescence.detection import CandidaDetector, CellExtractor")
print("detector = CandidaDetector()")
print("extractor = CellExtractor(detector, output_size=128)")
print("crops = extractor.extract('plate_image.jpg', threshold=0.5)")
print()
print("# Step 2: Convert to tensor")
print("tensor = extractor.crops_to_tensor(crops, normalize=True)")
print("# Shape: (N, 3, 128, 128)")
print()
print("# Step 3: Load Tendril VAE")
print("from candescence.tlv.factory import Factory")
print("from candescence.core.config import TLVConfig")
print("config = TLVConfig(experiment_name='analysis', save_name='detection_run')")
print("# ... configure and load trained model ...")
print()
print("# Step 4: Encode to latent space")
print("with torch.no_grad():")
print("    z, mu, logvar, skip = vae.encoder(tensor, cond)")
print()
print("# Step 5: Analyze latent space")
print("# - Cluster similar morphologies")
print("# - Visualize with PCA/UMAP")
print("# - Compare to known morphology grades")

## 9. Batch Processing

For large-scale analysis, use batch methods to process multiple images efficiently.

In [ ]:
# Batch processing example

print("Batch Processing API")
print("=" * 50)
print()
print("# Process multiple images")
print("image_paths = list(Path('plate_images/').glob('*.jpg'))")
print()
print("# Batch detection")
print("all_detections = detector.detect_batch(image_paths, threshold=0.5)")
print("# Returns: List[List[Detection]] - one list per image")
print()
print("# Batch extraction")
print("all_crops = extractor.extract_batch(image_paths, threshold=0.5)")
print("# Returns: List[List[CellCrop]] - one list per image")
print()
print("# Save crops to directory")
print("n_saved = extractor.extract_to_directory(")
print("    images=image_paths,")
print("    output_dir='extracted_cells/',")
print("    threshold=0.5,")
print("    filename_format='{source}_{idx}_{label}.png'")
print(")")
print("# Creates: extracted_cells/image1_0_Yeast_White.png, etc.")

## 10. Summary

This tutorial demonstrated:

1. **Morphology Classes**: 15 Candida morphology types organized into curriculum learning stages
2. **Detection**: FCOS-based object detection with the pre-trained production model
3. **Cell Extraction**: Cropping detected cells for downstream VAE analysis
4. **Statistics**: Analyzing detection results with class distributions and confidence metrics
5. **Visualization**: Plotting tools for detection analysis
6. **Integration**: Pipeline from plate images to VAE-ready tensors

### Key Classes

| Class | Purpose |
|-------|--------|
| `CandidaDetector` | FCOS inference wrapper |
| `CellExtractor` | Crop cells from detections |
| `MorphologyStats` | Statistical analysis |
| `DetectionVisualizer` | Plotting utilities |

### Next Steps

- Process your own plate images with the detector
- Extract cells and feed them to the Tendril VAE
- Analyze morphology distributions across conditions
- Train custom detection models with curriculum learning

In [ ]:
# Configuration summary
print("\n" + "=" * 50)
print("CONFIGURATION SUMMARY")
print("=" * 50)
print(f"\nProduction Model:")
print(f"  Config: <legacy-refined>/candescence_master/production/varasana/config.py")
print(f"  Weights: <legacy-refined>/candescence_master/production/varasana/model.pth")
print(f"\nArchitecture:")
print(f"  Detector: FCOS (Fully Convolutional One-Stage)")
print(f"  Backbone: ResNet-101")
print(f"  Neck: FPN (Feature Pyramid Network)")
print(f"  Classes: 15 morphology types")
print(f"\nTraining:")
print(f"  Strategy: Curriculum learning (6 stages)")
print(f"  Stages: white → opaque → gray → shmoo → pseudohyphae → hyphae")